# AST 与动态执行

学习目标：检查和转换小型语法树，读取函数与属性的信息，并在明确的命名空间中编译、执行可信源码，识别这些工具的适用边界。

前置知识：表达式与赋值、函数参数、字典与作用域、类和继承、property、异常处理、模块与文件路径。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

动态执行仅处理本章给定的可信源码。

配套脚本：位于 scripts/32-ast-and-execution/。

1. [source\_subject.py](scripts/32-ast-and-execution/source_subject.py)：提供学习时长函数，供 inspect 稳定读取签名与源码；只定义函数，无启动任务。

## 1 把源码解析为语法树

抽象语法树（abstract syntax tree，AST）用节点表达代码结构，适合检查“这里是赋值还是加法”，不需要先运行代码。ast.parse 解析源码，mode 决定接受的输入形式。

| mode 值 | 根节点原名 | 中文名称／含义 |
| --- | --- | --- |
| exec | ast.Module | 一组语句的根节点，body 是语句列表 |
| eval | ast.Expression | 单个表达式的根节点，body 是表达式节点 |

模式名称中的 exec 和 eval 在这里仅选择解析方式，不会调用同名执行函数。AST 节点结构可能随 Python 版本变化，本章按 3.12 编写。

In [1]:
import ast

module_tree = ast.parse("total = minutes + 5", mode="exec")
expression_tree = ast.parse("minutes + 5", mode="eval")

print(type(module_tree).__name__)  # Module：接受赋值语句。
print(type(expression_tree).__name__)  # Expression：接受单个表达式。
print(type(module_tree.body).__name__)  # list：可以容纳多条语句。
print(type(expression_tree.body).__name__)  # BinOp：本例表达式是二元运算。
# minutes 尚未提供值，仍然可以解析；这里没有进行名称查找或加法。

Module
Expression
list
BinOp


### 1.1 用 ast.dump 阅读节点

ast.dump 返回便于检查的树结构字符串；indent 控制缩进，默认不显示行列位置，include\_attributes=True 可补充这些属性。

| 节点原名 | 中文名称／含义 | 本例角色 |
| --- | --- | --- |
| ast.Assign | 赋值语句 | targets 保存赋值目标，value 保存右侧表达式 |
| ast.BinOp | 二元运算 | left 和 right 是两侧表达式，op 是运算符节点 |
| ast.Name | 名称 | id 是名称文本，ctx 表示读写等上下文 |
| ast.Load | 读取上下文 | 读取 minutes 的值 |
| ast.Store | 写入上下文 | 给 total 绑定值 |
| ast.Constant | 常量 | 保存数值 5 |
| ast.Add | 加法运算符 | minutes + 5 中的加法 |
| ast.Mult | 乘法运算符 | minutes × 5，在 Python 中写作 minutes \* 5 |

这里 minutes 表示学习分钟数，total 表示加上休息后的总分钟数。后面的树转换会把数值加法改成乘法，用来观察代码行为如何改变。

In [2]:
print(ast.dump(module_tree, indent=2))
# 沿 Assign.value 查看 BinOp，再沿 left、op、right 找到名称、加法与常量。
assignment = module_tree.body[0]
print(type(assignment.targets[0].ctx).__name__)  # Store：赋值目标。
print(type(assignment.value.left.ctx).__name__)  # Load：右侧读取名称。

Module(
  body=[
    Assign(
      targets=[
        Name(id='total', ctx=Store())],
      value=BinOp(
        left=Name(id='minutes', ctx=Load()),
        op=Add(),
        right=Constant(value=5)))],
  type_ignores=[])
Store
Load


## 2 遍历并检查节点

ast.walk 递归产生根节点及其所有后代，不保证遍历顺序。只关心有哪些节点时可以筛选；需要稳定显示时，自行排序。

下面只提取 Name 节点，并保留 ctx 区分读取与写入。名字出现过不等于它一定在某条运行路径上被使用，语法树检查本身不会执行程序。

In [3]:
name_usages = sorted(
    (node.id, type(node.ctx).__name__)
    for node in ast.walk(module_tree)
    if isinstance(node, ast.Name)
)
print(name_usages)  # [('minutes', 'Load'), ('total', 'Store')]。
assert name_usages == [("minutes", "Load"), ("total", "Store")]

[('minutes', 'Load'), ('total', 'Store')]


### 2.1 用 NodeVisitor 按节点类型处理

继承 ast.NodeVisitor 后，visit 会按节点类名查找对应方法，例如 BinOp 对应 visit\_BinOp；没有对应方法时使用 generic\_visit 继续访问子节点。

自定义方法接管节点后，必须自行访问子节点或调用 generic\_visit，否则其内部结构不会继续被访问。下面记录嵌套运算，最后排序显示，不依赖 ast.walk 的次序。

In [4]:
class OperatorCollector(ast.NodeVisitor):
    """收集表达式中所有二元运算符的节点名称。"""

    def __init__(self) -> None:
        self.operators: list[str] = []

    def visit_BinOp(self, node: ast.BinOp) -> None:
        """记录当前运算，并继续检查它的两个操作数。"""
        self.operators.append(type(node.op).__name__)
        self.generic_visit(node)


nested_tree = ast.parse("2 + (3 * 4)", mode="eval")
collector = OperatorCollector()
collector.visit(nested_tree)
print(sorted(collector.operators))  # ['Add', 'Mult']：内层乘法也被访问。
assert sorted(collector.operators) == ["Add", "Mult"]

['Add', 'Mult']


## 3 编译与执行是两个步骤

compile 把字符串或 AST 编译为代码对象，默认不会运行它。filename 是诊断用文件名；源码来自固定字符串时可以提供可辨认的名称，不需要创建同名文件。

| mode 值 | 中文名称／含义 | 本章的执行方式 |
| --- | --- | --- |
| eval | 单个表达式 | eval 返回表达式的值 |
| exec | 一组语句 | exec 执行语句，返回 None |
| single | 单个交互式输入 | exec 执行时会显示值不是 None 的表达式语句 |

表达式模式的 AST 根节点应为 Expression，语句模式应为 Module。下面只传一个命名空间字典，作为名称查找和赋值的位置；详细规则在后文展开。

In [5]:
compiled_expression = compile(expression_tree, "<lesson-expression>", "eval")
compiled_assignment = compile(module_tree, "<lesson-assignment>", "exec")
execution_scope = {"minutes": 20}

print("total" in execution_scope)  # False：编译没有执行赋值。
print(eval(compiled_expression, execution_scope))  # 25：执行时读取 minutes。
exec_result = exec(compiled_assignment, execution_scope)
print(exec_result, execution_scope["total"])  # None 25：结果通过字典读取。

interactive_code = compile("2 + 3", "<lesson-interactive>", "single")
exec(interactive_code, {})  # 5：single 模式显示这个表达式语句的值。

False
25
None 25


5

### 3.1 解析成功仍可能编译失败

ast.parse 不做全部作用域检查。函数外的 return 可以形成 AST，compile 仍会拒绝它；不能用“能得到树”代替“能编译”或“能正确运行”。

这里故意给出一个很小的反例，只捕获预期的 SyntaxError，使整篇能够继续执行。

In [6]:
outside_return = ast.parse("return 7", mode="exec")
print(type(outside_return.body[0]).__name__)  # Return：语法树已经生成。
try:
    compile(outside_return, "<outside-return>", "exec")
except SyntaxError as error:
    print(type(error).__name__, error.msg)
    # 预期 SyntaxError，消息指出 return 位于函数外。
    assert "outside function" in error.msg
else:
    raise AssertionError("函数外的 return 不应通过编译")

Return
SyntaxError 'return' outside function


## 4 受控转换后重新编译

ast.NodeTransformer 用访问方法的返回值替换原节点：返回原节点表示保留，返回新节点表示替换，返回 None 表示移除。遗漏 return 可能意外删掉节点；带子节点时仍需先处理子树。

下面只把本章固定整数表达式中的 Add 改为 Mult，其他运算保持原样。它用于演示行为改变，不是保持原意的优化，也不是任意代码的安全检查器。

新建表达式节点需要位置信息。ast.copy\_location 从原节点复制位置；ast.fix\_missing\_locations 递归为缺失位置的节点补上父节点位置。转换可能原地修改树，需要保留原树时应另行解析或复制。

In [7]:
class AddToMultiply(ast.NodeTransformer):
    """把二元加法改为乘法，保留其他二元运算。"""

    def visit_BinOp(self, node: ast.BinOp) -> ast.BinOp:
        """先转换子树，再替换当前加法节点。"""
        # 1. 先处理内层运算，避免漏掉嵌套加法。
        self.generic_visit(node)
        if not isinstance(node.op, ast.Add):
            return node

        # 2. 保留两侧表达式和来源位置，只改变运算符。
        replacement = ast.BinOp(
            left=node.left,
            op=ast.Mult(),
            right=node.right,
        )
        return ast.copy_location(replacement, node)


original_tree = ast.parse("2 + (3 + 4)", mode="eval")
changed_tree = AddToMultiply().visit(ast.parse("2 + (3 + 4)", mode="eval"))
changed_tree = ast.fix_missing_locations(changed_tree)
original_value = eval(compile(original_tree, "<original>", "eval"), {})
changed_value = eval(compile(changed_tree, "<changed>", "eval"), {})

print(original_value, changed_value)  # 9 24：两处加法都已改为乘法。
assert (original_value, changed_value) == (9, 24)
print(changed_tree.body.lineno == original_tree.body.lineno)  # True。

9 24
True


### 4.1 补齐位置不等于恢复原文

表达式与语句节点的 lineno 从 1 开始；col\_offset 是 UTF-8 字节偏移，从 0 开始，不一定等于中文字符串的字符下标。对应的结束位置可选。

compile 要求支持起始位置的节点具有 lineno 和 col\_offset。下面手工新建常量节点，观察补齐前后的差别。fix\_missing\_locations 只填充缺失位置，不重新推算真实源码范围；准确诊断需要转换过程维护位置。

In [8]:
manual_tree = ast.Expression(body=ast.Constant(value=9))
try:
    compile(manual_tree, "<missing-position>", "eval")
except TypeError as error:
    print(type(error).__name__)
    assert "lineno" in str(error)  # 预期缺少起始行号。
else:
    raise AssertionError("缺少位置的常量节点不应通过编译")

ast.fix_missing_locations(manual_tree)
print(ast.dump(manual_tree, include_attributes=True))
print(eval(compile(manual_tree, "<fixed-position>", "eval"), {}))  # 9。
# 本例位置来自补齐规则，不代表曾经存在一个同名源文件。

TypeError
Expression(body=Constant(value=9, lineno=1, col_offset=0, end_lineno=1, end_col_offset=0))
9


## 5 从语法树生成源码

ast.unparse 根据树生成源码，目标是重新解析后得到等价结构，不承诺文本与原文相同。空格、括号写法和普通注释不会按原样保留；不能把它当作无损源码编辑工具。

下面把再解析后的 ast.dump 与原树比较，默认不比较行列属性。这只检查本例树结构，不要求某种固定的排版。

In [9]:
formatted_source = "total=(2+3)  # 手工备注\n"
formatted_tree = ast.parse(formatted_source)
generated_source = ast.unparse(formatted_tree)
reparsed_tree = ast.parse(generated_source)

print(generated_source)  # total = 2 + 3：本例空格调整，普通注释消失。
print(generated_source == formatted_source)  # False：源码文本不相同。
print(ast.dump(reparsed_tree) == ast.dump(formatted_tree))  # True。
assert "手工备注" not in generated_source
assert ast.dump(reparsed_tree) == ast.dump(formatted_tree)

total = 2 + 3
False
True


## 6 自省已有函数

对象自省（introspection）读取运行时对象的信息；AST 检查源码结构，两者可以配合。inspect.signature 返回函数的调用签名，包括参数、默认值和标注。

下面用 runpy.run\_path 执行配套文件并取得其命名空间，再取出其中定义的函数。它也会执行代码，因此路径固定指向本章自写文件。函数只做整数计算；minutes 是每次学习分钟数，sessions 是次数，break\_minutes 是每次休息分钟数。

签名中的星号分隔仅限关键字形参。Signature.bind 检查实参能否按签名绑定，不会调用函数；Python 运行时不强制执行函数的参数类型标注。并非所有可调用对象都能提供签名。

In [10]:
import inspect
import runpy
from pathlib import Path

subject_path = Path("scripts/32-ast-and-execution/source_subject.py")
subject_scope = runpy.run_path(str(subject_path))
study_total = subject_scope["study_total"]
function_signature = inspect.signature(study_total)
print(function_signature)
# 签名中 break_minutes 为仅限关键字形参，默认值为 5。
bound_arguments = function_signature.bind(25, 2, break_minutes=5)
print(bound_arguments.arguments)
print(study_total(25, 2, break_minutes=5))  # 60：两次各含五分钟休息。

try:
    function_signature.bind(25, 2, 5)
except TypeError as error:
    print(type(error).__name__)  # TypeError：第三个值不能按位置传入。
else:
    raise AssertionError("仅限关键字形参不应接受位置实参")

(minutes: int, sessions: int, *, break_minutes: int = 5) -> int
{'minutes': 25, 'sessions': 2, 'break_minutes': 5}
60
TypeError


### 6.1 从函数源码回到 AST

inspect.getsource 返回对象的源码字符串。源码不可取得时可能抛出 OSError，内置函数等对象可能抛出 TypeError；动态创建的函数或交互式环境中的源码不保证可读取。

本例读取配套 .py 文件中定义的函数，再交给 ast.parse 和前面的 OperatorCollector。运行时对象因此可以成为源码检查的入口，但读取源码不等于执行源码。

In [11]:
function_source = inspect.getsource(study_total)
print(function_source)
source_collector = OperatorCollector()
source_collector.visit(ast.parse(function_source))
print(sorted(source_collector.operators))  # ['Add', 'Mult']。
assert sorted(source_collector.operators) == ["Add", "Mult"]
# 检查函数里的学习加休息、再乘次数；没有调用 study_total。

def study_total(
    minutes: int,
    sessions: int,
    *,
    break_minutes: int = 5,
) -> int:
    """计算每次学习与休息合计后的总分钟数。"""
    return (minutes + break_minutes) * sessions

['Add', 'Mult']


## 7 读取属性可能触发代码

getattr 按通常的属性访问规则取值，因此可能调用 property 的 getter。inspect.getattr\_static 避免描述器协议和动态属性查找，可能返回 property 对象本身；它也可能取不到动态生成的属性，不能替代普通属性读取。

property 已用于管理属性；这里关注的是检查对象时是否触发它的读取逻辑。下例故意累计读取次数，使副作用可以直接核对。

In [12]:
class ReadingCounter:
    """记录 value 属性的 getter 被调用了几次。"""

    def __init__(self) -> None:
        self.reads = 0

    @property
    def value(self) -> int:
        """累计一次读取，再返回固定数值。"""
        self.reads += 1
        return 12


reading_counter = ReadingCounter()
static_attribute = inspect.getattr_static(reading_counter, "value")
print(isinstance(static_attribute, property), reading_counter.reads)
# True 0：得到 property 对象，getter 尚未执行。
print(getattr(reading_counter, "value"), reading_counter.reads)
# 12 1：普通读取调用了 getter。
assert isinstance(static_attribute, property)
assert reading_counter.reads == 1

True 0
12 1


## 8 明确 eval 与 exec 的命名空间

Python 3.12 中，eval 的 globals 参数接收字典，locals 接收映射。都省略时使用调用处环境；只提供 globals 时，它也作为 locals。直接读取名称时，局部命名空间可以覆盖全局同名项。

本章显式传字典，避免把动态执行与 Notebook 里已有的名称混在一起。下面复用一个表达式代码对象，只改变查找用的命名空间。

In [13]:
namespace_code = compile("minutes + extra", "<namespace-expression>", "eval")
global_scope = {"minutes": 20, "extra": 5}
local_scope = {"minutes": 40}

print(eval(namespace_code, global_scope, local_scope))  # 45：局部 40 加全局 5。
print(eval(namespace_code, global_scope))  # 25：同一字典兼作局部命名空间。
print(global_scope["minutes"], local_scope["minutes"])  # 20 40：输入值未改。

45
25
20 40


### 8.1 用同一个字典执行语句并读取结果

exec 只提供 globals 时，这个参数必须是普通字典，并同时用于全局和局部命名空间。执行产生的绑定可以从该字典读取；exec 自身返回 None。

在函数内部使用 exec 时，不要依赖修改默认局部命名空间来改变函数局部变量。需要读取执行结果，就像下面这样显式传入并保留字典。3.12 的 exec 中这两个命名空间参数按位置传递。

In [14]:
statement_code = compile(
    "total = minutes + 5\ndoubled = total * 2",
    "<namespace-statements>",
    "exec",
)
statement_scope = {"minutes": 20}
exec(statement_code, statement_scope)
print(statement_scope["total"], statement_scope["doubled"])  # 25 50。
assert statement_scope["total"] == 25
assert statement_scope["doubled"] == 50
# 结果保存在显式字典中，不靠 exec 的返回值取得。

25 50


### 8.2 分开两个字典会改变函数内的查找

exec 接收不同的 globals 与 locals 时，按类似类定义体的规则执行。顶层赋值和函数定义落在局部命名空间中；其中定义的函数查找全局名称时使用 globals，不会把这个单独的 locals 自动变成闭包。

这使“执行时可以直接看到一个名称”和“新定义的函数稍后能看到同一个名称”成为两件事。普通用法优先只传一个字典；这里分开传入是为了观察边界。

In [15]:
function_code = compile(
    "base = 6\ndef score():\n    return base + 2\n",
    "<separate-namespaces>",
    "exec",
)
function_globals = {"base": 20}
function_locals = {}
exec(function_code, function_globals, function_locals)

print(function_locals["base"])  # 6：顶层赋值落在 locals。
print(function_locals["score"]())  # 22：函数里的 base 来自 globals。
assert function_globals["base"] == 20
assert function_locals["score"]() == 22

6
22


## 9 内置名称与可信输入边界

如果传入的 globals 没有 \_\_builtins\_\_ 键，eval 和 exec 都会插入 builtins 模块的字典。因此，传空字典不等于禁用内置函数；预先提供这个键可以选择可用的内置名称。

这种选择和命名空间字典用于控制名称解析，不构成安全沙箱。eval 可以执行表达式中的函数调用，exec 可以执行语句；本章只使用固定、可信的源码与对象，不将它们用作外部输入处理器。

In [16]:
eval_scope = {}
print(eval("len([2, 3])", eval_scope))  # 2：空字典仍能找到内置 len。
print("__builtins__" in eval_scope)  # True：eval 已插入内置命名空间。
exec_scope = {}
exec("total = len([2, 3, 4])", exec_scope)
print(exec_scope["total"], "__builtins__" in exec_scope)  # 3 True。

selected_builtins = {"__builtins__": {"len": len}}
print(eval("len([2, 3])", selected_builtins))  # 2：显式提供需要的名称。
# 这里只观察名称来源，不把内置名称筛选当作安全隔离。
assert selected_builtins["__builtins__"] == {"len": len}

2
True
3 True
2


## 10 只读取字面值时使用 literal\_eval

ast.literal\_eval 读取 Python 字面值和容器显示，例如数字、字符串、列表、元组、字典、集合、布尔值和 None；也支持 bytes 与 Ellipsis。它不执行任意 Python 代码，不查找变量，也不支持一般函数调用、下标或任意算术表达式。

它仍不是无条件安全的数据入口：输入的大小与嵌套复杂度可能导致内存、C 栈或 CPU 资源耗尽，形成拒绝服务（denial of service，DoS）。官方不建议对不可信数据调用它。本例只观察很小的固定文本，不运行资源耗尽实验。

In [17]:
literal_text = "{'minutes': [20, 30], 'enabled': True, 'note': None}"
literal_settings = ast.literal_eval(literal_text)
print(literal_settings)
assert literal_settings["minutes"] == [20, 30]
assert literal_settings["note"] is None

for rejected_text in ["2 + 3", "minutes", "len([1])"]:
    try:
        ast.literal_eval(rejected_text)
    except ValueError as error:
        print(rejected_text, type(error).__name__)
        # 本组均应为 ValueError：分别是一般算术、名称与函数调用。
    else:
        raise AssertionError(f"本例不应接受：{rejected_text}")

{'minutes': [20, 30], 'enabled': True, 'note': None}
2 + 3 ValueError
minutes ValueError
len([1]) ValueError


## 本章小结

（1）AST 表达代码结构；解析、编译、执行是不同阶段，任何前一步成功都不能代替后一步验证。

（2）访问器负责继续遍历子节点；转换器用返回值替换节点，新增节点还要维护或补齐位置。unparse 不保留原文排版。

（3）inspect 可以连接对象与源码；普通属性读取可能触发 getter，静态读取可能返回描述器本身。

（4）执行时显式安排命名空间并读取结果；字典和内置名称筛选不构成沙箱，literal\_eval 也有输入复杂度限制。

自查：能否分别说明某个工具是在处理源码结构、已有对象、名称绑定，还是执行代码？

## 练习

（1）先预测下面四行输出，再运行核对。解释 exec 模式根节点中的 body、eval 模式根节点中的 body，以及解析时为什么不需要为 minutes 提供值。核对标准是四行预测与输出一致，且能区分树结构与执行结果。

In [18]:
exercise_module = ast.parse("total = minutes + 5", mode="exec")
exercise_expression = ast.parse("minutes + 5", mode="eval")
print(type(exercise_module).__name__)
print(type(exercise_module.body[0].value.op).__name__)
print(type(exercise_expression.body).__name__)
print(eval(compile(exercise_expression, "<exercise>", "eval"), {"minutes": 7}))
# 先记录预测，运行后逐项核对；不要把节点类型与数值混为一谈。

Module
Add
BinOp
12


（2）对固定表达式 (2 + 3) + (4 - 1) 使用 AddToMultiply，另行解析以保留原树。补齐位置后分别编译执行，检查原值为 8、转换值为 18。

再用 OperatorCollector 检查转换后恰有两个 Mult、一个 Sub；把转换树用 unparse 生成源码并重新解析，检查它与转换树的 ast.dump 一致。这里减法中的两个操作数是整数 4 和 1，转换不应改变它们的运算符。

In [19]:
exercise_transform_source = "(2 + 3) + (4 - 1)"
# 在此分别解析原树和待转换树，比较执行值、运算符数量与再解析结构。

（3）复用前文 function\_code，先用一个包含 base=20 的字典执行，再用 globals 中 base=20、locals 为空的两个字典执行。检查两次执行后局部绑定的 base 都为 6，但 score 的调用结果分别为 8 与 22。

继续只把第二次执行的 globals 中 base 改为 30，不重新编译和定义函数，检查 score 改为返回 32，而 locals 中 base 仍为 6。解释函数查找全局名称的时间和位置；两种传参方式都不能用于隔离不可信代码。

In [20]:
# 在此创建彼此独立的字典，执行同一个 function_code 并检查函数结果。
# 只使用前文固定源码；最后一次观察仅更新第二组 globals 的 base。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | AST：[模块与版本边界](https://docs.python.org/3.12/library/ast.html)、[Module](https://docs.python.org/3.12/library/ast.html#ast.Module)、[Expression](https://docs.python.org/3.12/library/ast.html#ast.Expression)、[Assign](https://docs.python.org/3.12/library/ast.html#ast.Assign)、[Name 与读写上下文](https://docs.python.org/3.12/library/ast.html#ast.Name)、[Constant](https://docs.python.org/3.12/library/ast.html#ast.Constant)、[BinOp 与运算符](https://docs.python.org/3.12/library/ast.html#ast.BinOp)、[parse 与编译边界](https://docs.python.org/3.12/library/ast.html#ast.parse)、[dump](https://docs.python.org/3.12/library/ast.html#ast.dump)、[walk](https://docs.python.org/3.12/library/ast.html#ast.walk)、[NodeVisitor](https://docs.python.org/3.12/library/ast.html#ast.NodeVisitor)、[NodeTransformer](https://docs.python.org/3.12/library/ast.html#ast.NodeTransformer)、[copy\_location](https://docs.python.org/3.12/library/ast.html#ast.copy_location)、[fix\_missing\_locations](https://docs.python.org/3.12/library/ast.html#ast.fix_missing_locations)、[行号与 UTF-8 列偏移](https://docs.python.org/3.12/library/ast.html#ast.AST.lineno)、[unparse](https://docs.python.org/3.12/library/ast.html#ast.unparse)、[literal\_eval 与资源风险](https://docs.python.org/3.12/library/ast.html#ast.literal_eval)。对象自省：[signature](https://docs.python.org/3.12/library/inspect.html#inspect.signature)、[Signature.bind](https://docs.python.org/3.12/library/inspect.html#inspect.Signature.bind)、[getsource](https://docs.python.org/3.12/library/inspect.html#inspect.getsource)、[静态属性读取](https://docs.python.org/3.12/library/inspect.html#fetching-attributes-statically)、[getattr\_static 的限制](https://docs.python.org/3.12/library/inspect.html#inspect.getattr_static)、[getattr](https://docs.python.org/3.12/library/functions.html#getattr)、[property](https://docs.python.org/3.12/library/functions.html#property)、[runpy.run\_path](https://docs.python.org/3.12/library/runpy.html#runpy.run_path)。编译执行：[compile 的模式](https://docs.python.org/3.12/library/functions.html#compile)、[eval 与内置名称](https://docs.python.org/3.12/library/functions.html#eval)、[exec 与命名空间](https://docs.python.org/3.12/library/functions.html#exec)、[动态特性中的名称查找](https://docs.python.org/3.12/reference/executionmodel.html#interaction-with-dynamic-features)、[类型标注不强制运行时检查](https://docs.python.org/3.12/library/typing.html)。 |